In [ ]:
from google.colab import files
uploaded = files.upload()

KeyboardInterrupt: 

In [ ]:
import pandas as pd
import io

dataframes = {}
for filename, content in uploaded.items():
    df_name = filename.split('.')[0] # Use filename without extension as key
    dataframes[df_name] = pd.read_csv(io.StringIO(content.decode('utf-8')))

print("Files successfully read into DataFrames. Here's the head of the first one:")
# Display the head of one of the dataframes to confirm
if dataframes:
    first_df_name = list(dataframes.keys())[1]
    print(f"DataFrame '{first_df_name}':\n{dataframes[first_df_name].head()}")

In [ ]:
#the table is merged_df
merged_df = None

for df_name, df in dataframes.items():
    # Standardize column names before merging
    if 'Stn_no' in df.columns:
        df = df.rename(columns={'Stn_no': 'stn_no'})

    if merged_df is None:
        # Initialize merged_df with the first dataframe
        merged_df = df.copy()
    else:
        # Merge subsequent dataframes on 'Month' and 'stn_no'
        merged_df = pd.merge(merged_df, df, on=['Month', 'stn_no'], how='outer')

print("Merged DataFrame created successfully. Here's the head:")
display(merged_df.head())

In [ ]:
import pandas as pd
import numpy as np

def calculate_wqi(row, weights=None):
    # Default equal weights if not provided
    if weights is None:
        weights = {
            'PH': 1/6,
            'EC': 1/6,
            'SAR': 1/6,
            'RSC': 1/6,
            'Ca_Mg': 1/6,
            'SO4_CL': 1/6
        }

    # Ensure weights sum to 1 for scaling to 100
    total_weight = sum(weights.values())
    if abs(total_weight - 1.0) > 1e-6: # Check if sum is close to 1
        # print(f"Warning: Weights do not sum to 1. Rescaling. Original sum: {total_weight}") # Removed print for cleaner output
        weights = {k: v / total_weight for k, v in weights.items()}

    # 1. Continuous Quality Score (r_param) Calculation (0-1 range)
    # These conditions define how 'good' each parameter is on a continuous scale.
    # Using np.clip to ensure scores stay within 0 and 1.
    #linear increase is for nuanced
    # PH (Optimal: 6.5 - 8.5)
    if 6.5 <= row['PH'] <= 8.5: r_ph = 1.0
    elif row['PH'] < 5.5: r_ph = 0.0
    elif row['PH'] > 9.5: r_ph = 0.0
    elif row['PH'] < 6.5: r_ph = (row['PH'] - 5.5) / (6.5 - 5.5) # Linear increase from 5.5 to 6.5
    else: r_ph = (9.5 - row['PH']) / (9.5 - 8.5) # Linear decrease from 8.5 to 9.5
    r_ph = np.clip(r_ph, 0, 1)

    # EC (Optimal: <= 0.7)
    if row['EC'] <= 0.7: r_ec = 1.0
    elif row['EC'] >= 2.0: r_ec = 0.0
    else: r_ec = (2.0 - row['EC']) / (2.0 - 0.7) # Linear decrease from 0.7 to 2.0
    r_ec = np.clip(r_ec, 0, 1)

    # SAR (Optimal: <= 0.5, tighter boundary)
    if row['SAR'] <= 0.5: r_sar = 1.0
    elif row['SAR'] >= 2.0: r_sar = 0.0
    else: r_sar = (2.0 - row['SAR']) / (2.0 - 0.5) # Linear decrease from 0.5 to 2.0
    r_sar = np.clip(r_sar, 0, 1)

    # RSC (Optimal: <= -12.0, tighter boundary)
    if row['RSC'] <= -12.0: r_rsc = 1.0
    elif row['RSC'] >= -4.0: r_rsc = 0.0
    else: r_rsc = (-4.0 - row['RSC']) / (-4.0 - (-12.0)) # Linear decrease from -12.0 to -4.0
    r_rsc = np.clip(r_rsc, 0, 1)

    # Ca_Mg (Optimal: >= 5.0, broader window)
    if row['Ca_Mg'] >= 5.0: r_camg = 1.0
    elif row['Ca_Mg'] <= 0.5: r_camg = 0.0
    else: r_camg = (row['Ca_Mg'] - 0.5) / (5.0 - 0.5) # Linear increase from 0.5 to 5.0
    r_camg = np.clip(r_camg, 0, 1)

    # SO4_CL (Optimal: >= 0.5, based on observed data range and need for differentiation)
    if row['SO4_CL'] >= 0.5: r_so4cl = 1.0
    elif row['SO4_CL'] <= 0.1: r_so4cl = 0.0
    else: r_so4cl = (row['SO4_CL'] - 0.1) / (0.5 - 0.1) # Linear increase from 0.1 to 0.5
    r_so4cl = np.clip(r_so4cl, 0, 1)

    # 2. Weighted Score Calculation
    weighted_score = (
        r_ph * weights['PH'] +
        r_ec * weights['EC'] +
        r_sar * weights['SAR'] +
        r_rsc * weights['RSC'] +
        r_camg * weights['Ca_Mg'] +
        r_so4cl * weights['SO4_CL']
    )

    # Scale the score to a 0-100 range
    final_score = weighted_score * 100

    # 3. Categories (G, P, VP) - based on the scaled WQI (can be adjusted as needed)
    if final_score >= 85:
        cat = 'G'
    elif final_score >= 60:
        cat = 'P'
    else:
        cat = 'VP'

    return pd.Series([final_score, cat])

# NOTE: We are no longer applying the function here directly with default weights.
# The application will be done in a new cell to allow for custom weight definition.

print("calculate_wqi function updated for continuous WQI scoring!")

### Applying Custom Weights to WQI Calculation

Based on the IS standards highlighting EC, SAR, and RSC as three main parameters, we will now re-calculate the WQI using custom weights. This will ensure these parameters have a more significant impact on the overall WQI score, aligning the calculation with your domain expertise.

We will assign higher weights to EC, SAR, and RSC, and proportionally lower weights to the other parameters, ensuring the total sum of weights is 1.

In [ ]:
# Define custom weights based on domain knowledge
# Example: Giving higher weights to EC, SAR, RSC, and lower to others.
# Ensure weights sum to 1.
custom_weights = {
    'EC': 0.3,      # Increased importance
    'SAR': 0.2,     # Increased importance
    'RSC': 0.2,     # Increased importance
    'PH': 0.1,      # Lower importance
    'Ca_Mg': 0.1,   # Lower importance
    'SO4_CL': 0.1   # Lower importance
}

print("Custom weights defined:")
for param, weight in custom_weights.items():
    print(f"  {param}: {weight}")

# Apply the updated calculate_wqi function with custom weights
# We'll create a new temporary WQI and WQI_Cat columns to compare
# or overwrite the existing ones if preferred.

# Create a copy to avoid modifying the original 'merged_df' if you want to keep the original WQI
# For this demonstration, we'll overwrite it to update all subsequent analyses.
merged_df[['WQI', 'WQI_Cat']] = merged_df.apply(lambda row: calculate_wqi(row, weights=custom_weights), axis=1)

print("\nWQI and Categories Generated with Custom Weights!")
print(merged_df['WQI_Cat'].value_counts())

# Update X_full and y_full for subsequent model training and analysis
X_full = merged_df.drop(['WQI', 'WQI_Cat', 'stn_no', 'Month'], axis=1)
y_full = merged_df['WQI']

print("\nX_full and y_full updated with new WQI calculation.")

### Re-evaluating Parameter Importance with Custom Weighted WQI

Now that the WQI calculation incorporates custom weights, let's re-run the correlation analysis and the linear regression models to see how the importance of EC, SAR, and RSC has changed. We expect to see their influence on the WQI reflected more strongly in these metrics.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# --- Classification Split ---
# Filter out the 'G' class for classification as it has only one sample
# making it impossible to perform a meaningful train/test split or classification.
classification_df = merged_df[merged_df['WQI_Cat'] != 'G'].copy()

# Define features (X_clf) and target (y_clf) for classification
X_clf = classification_df.drop(['WQI', 'WQI_Cat', 'stn_no', 'Month'], axis=1)
y_classification = classification_df['WQI_Cat']

# Encode the categorical target variable (to convert good,poor into n.s so it can communicate)
label_encoder = LabelEncoder()
y_encoded_classification = label_encoder.fit_transform(y_classification)

# Split the data into training and testing sets for classification (70-30 split)
# Removed 'stratify' parameter as we are explicitly handling class imbalance by removing 'G'.
X_train_clf, X_test_clf, y_train_clf, y_test_clf = train_test_split(
    X_clf, y_encoded_classification, test_size=0.3, random_state=42
)

print("--- Data split for Classification (70-30) ---")
print(f"X_train_clf shape: {X_train_clf.shape}")
print(f"X_test_clf shape: {X_test_clf.shape}")
print(f"y_train_clf shape: {y_train_clf.shape}")
print(f"y_test_clf shape: {y_test_clf.shape}")

print("\nOriginal WQI_Cat classes (used for classification):", label_encoder.classes_)
print("Encoded WQI_Cat classes (0, 1, 2, etc.):", label_encoder.transform(label_encoder.classes_))

# --- Regression Split ---
# The regression task uses the full dataset including the 'G' sample, as it's a continuous target.
# Define features (X_reg) and target (y_reg) for regression using the original full dataset
X_reg = merged_df.drop(['WQI', 'WQI_Cat', 'stn_no', 'Month'], axis=1)
y_regression = merged_df['WQI']

# Split the data into training and testing sets for regression (70-30 split)
# No stratification for continuous target variables
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_regression, test_size=0.3, random_state=42 # 42 is just a no.
)

print("\n--- Data split for Regression (70-30) ---")
print(f"X_train_reg shape: {X_train_reg.shape}")
print(f"X_test_reg shape: {X_test_reg.shape}")
print(f"y_train_reg shape: {y_train_reg.shape}")
print(f"y_test_reg shape: {y_test_reg.shape}")

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

# Calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero: add a small epsilon to avoid NaN or Inf where y_true is 0
    return np.mean(np.abs((y_true - y_pred) / (y_true + np.finfo(float).eps))) * 100

# Prepare the full dataset for cross-validation
# Define features (X_full) directly using merged_df, assuming merged_df is already defined.
# Drop 'WQI' as 'WQI_Cat' is derived from it, and 'stn_no' and 'Month' are identifiers
X_full = merged_df.drop(['WQI', 'WQI_Cat', 'stn_no', 'Month'], axis=1)

# Target for regression (continuous WQI)
y_full = merged_df['WQI']

n_splits = 5 # Number of folds 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

print(f"\n--- Performing {n_splits}-Fold Cross-Validation for Linear Regression ---")
mse_scores = []
rmse_scores = []
mae_scores = []
mape_scores = []
r2_scores = []

# Re-initialize the Linear Regression model for each fold to ensure a fresh start
for fold, (train_index, test_index) in enumerate(kf.split(X_full)):
    X_train_fold, X_test_fold = X_full.iloc[train_index], X_full.iloc[test_index]
    y_train_fold, y_test_fold = y_full.iloc[train_index], y_full.iloc[test_index]

    linear_reg_model = LinearRegression()
    linear_reg_model.fit(X_train_fold, y_train_fold)
    y_pred_fold = linear_reg_model.predict(X_test_fold)

    mse_scores.append(mean_squared_error(y_test_fold, y_pred_fold))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test_fold, y_pred_fold)))
    mae_scores.append(mean_absolute_error(y_test_fold, y_pred_fold))
    mape_scores.append(mean_absolute_percentage_error(y_test_fold, y_pred_fold))
    r2_scores.append(r2_score(y_test_fold, y_pred_fold))

# Calculate and print average metrics
print(f"  Average MSE: {np.mean(mse_scores):.2f} (Std: {np.std(mse_scores):.2f})")
print(f"  Average RMSE: {np.mean(rmse_scores):.2f} (Std: {np.std(rmse_scores):.2f})")
print(f"  Average MAE: {np.mean(mae_scores):.2f} (Std: {np.std(mae_scores):.2f})")
print(f"  Average MAPE: {np.mean(mape_scores):.2f}% (Std: {np.std(mape_scores):.2f}%)")
print(f"  Average R2 Score: {np.mean(r2_scores):.2f} (Std: {np.std(r2_scores):.2f})")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.linear_model import LinearRegression

# Train a Linear Regression model on the full dataset with the NEW WQI
# X_full and y_full have been updated with the new WQI calculation.
linear_reg_final_new_wqi = LinearRegression()
linear_reg_final_new_wqi.fit(X_full, y_full)

# Get the coefficients (feature importances for linear models)
coefficients_lr_new_wqi = linear_reg_final_new_wqi.coef_

# Create a pandas Series for better visualization
features_coef_lr_new_wqi = pd.Series(coefficients_lr_new_wqi, index=X_full.columns)

# Sort the features by the absolute value of their coefficients
sorted_features_coef_lr_new_wqi = features_coef_lr_new_wqi.abs().sort_values(ascending=False)

print("Absolute Feature Coefficients for Linear Regression (NEW WQI Prediction - Custom Weights):")
print(sorted_features_coef_lr_new_wqi)

# Plot the feature coefficients
plt.figure(figsize=(10, 6))
sns.barplot(x=sorted_features_coef_lr_new_wqi.values, y=sorted_features_coef_lr_new_wqi.index, palette='plasma')
plt.title('Absolute Feature Coefficients for NEW WQI Prediction (Linear Regression - Custom Weights)')
plt.xlabel('Absolute Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
from sklearn.linear_model import BayesianRidge
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

# Calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero: add a small epsilon to avoid NaN or Inf where y_true is 0
    return np.mean(np.abs((y_true - y_pred) / (y_true + np.finfo(float).eps))) * 100

# Prepare the full dataset for cross-validation
# Define features (X_full) directly using merged_df, assuming merged_df is already defined.
# Drop 'WQI' as 'WQI_Cat' is derived from it, and 'stn_no' and 'Month' are identifiers
X_full = merged_df.drop(['WQI', 'WQI_Cat', 'stn_no', 'Month'], axis=1)

# Target for regression (continuous WQI)
y_full = merged_df['WQI']

n_splits = 5 # Number of folds - Changed from 10 to 5
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

print(f"\n--- Performing {n_splits}-Fold Cross-Validation for Bayesian Ridge Regression ---")
mse_scores = []
rmse_scores = []
mae_scores = []
mape_scores = []
r2_scores = []

# Re-initialize the Bayesian Ridge model for each fold to ensure a fresh start
for fold, (train_index, test_index) in enumerate(kf.split(X_full)):
    X_train_fold, X_test_fold = X_full.iloc[train_index], X_full.iloc[test_index]
    y_train_fold, y_test_fold = y_full.iloc[train_index], y_full.iloc[test_index]

    bayesian_ridge_model = BayesianRidge()
    bayesian_ridge_model.fit(X_train_fold, y_train_fold)
    y_pred_fold = bayesian_ridge_model.predict(X_test_fold)

    mse_scores.append(mean_squared_error(y_test_fold, y_pred_fold))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test_fold, y_pred_fold)))
    mae_scores.append(mean_absolute_error(y_test_fold, y_pred_fold))
    mape_scores.append(mean_absolute_percentage_error(y_test_fold, y_pred_fold))
    r2_scores.append(r2_score(y_test_fold, y_pred_fold))

# Calculate and print average metrics
print(f"  Average MSE: {np.mean(mse_scores):.2f} (Std: {np.std(mse_scores):.2f})")
print(f"  Average RMSE: {np.mean(rmse_scores):.2f} (Std: {np.std(rmse_scores):.2f})")
print(f"  Average MAE: {np.mean(mae_scores):.2f} (Std: {np.std(mae_scores):.2f})")
print(f"  Average MAPE: {np.mean(mape_scores):.2f}% (Std: {np.std(mape_scores):.2f}%)")
print(f"  Average R2 Score: {np.mean(r2_scores):.2f} (Std: {np.std(r2_scores):.2f})")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.linear_model import BayesianRidge

# Train a Bayesian Ridge model on the full dataset with the NEW WQI
# X_full and y_full have been updated with the new WQI calculation.
bayesian_ridge_final_new_wqi = BayesianRidge()
bayesian_ridge_final_new_wqi.fit(X_full, y_full)

# Get the coefficients (feature importances for linear models)
coefficients_br_new_wqi = bayesian_ridge_final_new_wqi.coef_

# Create a pandas Series for better visualization
features_coef_br_new_wqi = pd.Series(coefficients_br_new_wqi, index=X_full.columns)

# Sort the features by the absolute value of their coefficients
sorted_features_coef_br_new_wqi = features_coef_br_new_wqi.abs().sort_values(ascending=False)

print("Feature Coefficients for Bayesian Ridge Regressor (NEW WQI Prediction - Custom Weights):")
print(sorted_features_coef_br_new_wqi)

# Plot the feature coefficients
plt.figure(figsize=(10, 6))
sns.barplot(x=sorted_features_coef_br_new_wqi.values, y=sorted_features_coef_br_new_wqi.index, palette='viridis')
plt.title('Absolute Feature Coefficients for NEW WQI Prediction (Bayesian Ridge Regressor - Custom Weights)')
plt.xlabel('Absolute Coefficient Value')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel as C
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, make_scorer
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero: add a small epsilon to avoid NaN or Inf where y_true is 0
    return np.mean(np.abs((y_true - y_pred) / (y_true + np.finfo(float).eps))) * 100

# Prepare the full dataset for cross-validation
# Define features (X_full) directly using merged_df, assuming merged_df is already defined.
# Drop 'WQI' as 'WQI_Cat' is derived from it, and 'stn_no' and 'Month' are identifiers
X_full = merged_df.drop(['WQI', 'WQI_Cat', 'stn_no', 'Month'], axis=1)

# Target for regression (continuous WQI)
y_full = merged_df['WQI']

n_splits = 5 # Number of folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

print(f"\n--- Performing {n_splits}-Fold Cross-Validation for Gaussian Process Regressor with Optimized Parameters ---")

# Define the pipeline for scaling and GPR with the best parameters found previously
# Best parameters were:
#   'gpr__alpha': 0.1
#   'gpr__kernel__k1__constant_value': 0.932513339062763
#   'gpr__kernel__k2__length_scale': 0.08168445784103573
#   'gpr__n_restarts_optimizer': 10
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('gpr', GaussianProcessRegressor(
        kernel=C(0.932513339062763) * RBF(0.08168445784103573),
        alpha=0.1,
        n_restarts_optimizer=10,
        random_state=42
    ))
])

# Assign the configured pipeline as the best_gpr_model for evaluation
best_gpr_model = pipe

mse_scores = []
rmse_scores = []
mae_scores = []
mape_scores = []
r2_scores = []

# Evaluate the best model using the same KFold strategy to get all metrics and their standard deviations
for fold, (train_index, test_index) in enumerate(kf.split(X_full)):
    X_train_fold, X_test_fold = X_full.iloc[train_index], X_full.iloc[test_index]
    y_train_fold, y_test_fold = y_full.iloc[train_index], y_full.iloc[test_index]

    # The best_gpr_model is a Pipeline, so it handles scaling internally
    # We fit it on each fold's training data and predict on test data
    best_gpr_model.fit(X_train_fold, y_train_fold)
    y_pred_fold = best_gpr_model.predict(X_test_fold)

    mse_scores.append(mean_squared_error(y_test_fold, y_pred_fold))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test_fold, y_pred_fold)))
    mae_scores.append(mean_absolute_error(y_test_fold, y_pred_fold))
    mape_scores.append(mean_absolute_percentage_error(y_test_fold, y_pred_fold))
    r2_scores.append(r2_score(y_test_fold, y_pred_fold))

# Calculate and print average metrics for the best model
print(f"\n--- Performance of GPR Model with Fixed Optimized Parameters and {n_splits}-Fold Cross-Validation ---")
print(f"  Average MSE: {np.mean(mse_scores):.2f} (Std: {np.std(mse_scores):.2f})")
print(f"  Average RMSE: {np.mean(rmse_scores):.2f} (Std: {np.std(rmse_scores):.2f})")
print(f"  Average MAE: {np.mean(mae_scores):.2f} (Std: {np.std(mae_scores):.2f})")
print(f"  Average MAPE: {np.mean(mape_scores):.2f}% (Std: {np.std(mape_scores):.2f}%)")
print(f"  Average R2 Score: {np.mean(r2_scores):.2f} (Std: {np.std(r2_scores):.2f})")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.inspection import permutation_importance

# Ensure X_full and y_full are defined (as they should be from previous cells)
# X_full = merged_df.drop(['WQI', 'WQI_Cat', 'stn_no', 'Month'], axis=1)
# y_full = merged_df['WQI']

# The 'best_gpr_model' pipeline was defined in cell s6izQNTHlpVK
# and is already fitted on the full dataset during its cross-validation step,
# but we'll refit it here to ensure it's using the full X_full, y_full for this calculation
# if it wasn't the last fit within the CV loop.

best_gpr_model.fit(X_full, y_full)

# Calculate Permutation Feature Importance
# n_repeats: number of times to permute a feature
# random_state for reproducibility
# n_jobs=-1 uses all available CPU cores for faster computation
result = permutation_importance(
    best_gpr_model, X_full, y_full, n_repeats=10, random_state=42, n_jobs=-1
)

# Organize the results into a pandas Series
sorted_importances_gpr = pd.Series(
    result.importances_mean, index=X_full.columns
).sort_values(ascending=False)

print("Permutation Feature Importances for GPR (WQI Prediction):")
print(sorted_importances_gpr)

# Plot the feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x=sorted_importances_gpr.values, y=sorted_importances_gpr.index, palette='crest')
plt.title('Permutation Feature Importance for WQI Prediction (Gaussian Process Regressor)')
plt.xlabel('Mean Decrease in R2 Score (Importance)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


In [ ]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd

# Calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero: add a small epsilon to avoid NaN or Inf where y_true is 0
    return np.mean(np.abs((y_true - y_pred) / (y_true + np.finfo(float).eps))) * 100

# Prepare the full dataset for cross-validation
# Define features (X_full) directly using merged_df, assuming merged_df is already defined.
# Drop 'WQI' as 'WQI_Cat' is derived from it, and 'stn_no' and 'Month' are identifiers
X_full = merged_df.drop(['WQI', 'WQI_Cat', 'stn_no', 'Month'], axis=1)

# Target for regression (continuous WQI)
y_full = merged_df['WQI']

n_splits = 5 # Number of folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

print(f"\n--- Performing {n_splits}-Fold Cross-Validation for Random Forest Regressor ---")
mse_scores = []
rmse_scores = []
mae_scores = []
mape_scores = []
r2_scores = []

# Perform K-Fold Cross-Validation
for fold, (train_index, test_index) in enumerate(kf.split(X_full)):
    X_train_fold, X_test_fold = X_full.iloc[train_index], X_full.iloc[test_index]
    y_train_fold, y_test_fold = y_full.iloc[train_index], y_full.iloc[test_index]

    # Initialize the Random Forest Regressor model for each fold
    rf_reg_model = RandomForestRegressor(n_estimators=100, random_state=42)

    # Train the model
    rf_reg_model.fit(X_train_fold, y_train_fold)

    # Make predictions
    y_pred_fold = rf_reg_model.predict(X_test_fold)

    # Evaluate the model for this fold
    mse_scores.append(mean_squared_error(y_test_fold, y_pred_fold))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test_fold, y_pred_fold)))
    mae_scores.append(mean_absolute_error(y_test_fold, y_pred_fold))
    mape_scores.append(mean_absolute_percentage_error(y_test_fold, y_pred_fold))
    r2_scores.append(r2_score(y_test_fold, y_pred_fold))

# Calculate and print average metrics across all folds
print(f"  Average MSE: {np.mean(mse_scores):.2f} (Std: {np.std(mse_scores):.2f})")
print(f"  Average RMSE: {np.mean(rmse_scores):.2f} (Std: {np.std(rmse_scores):.2f})")
print(f"  Average MAE: {np.mean(mae_scores):.2f} (Std: {np.std(mae_scores):.2f})")
print(f"  Average MAPE: {np.mean(mape_scores):.2f}% (Std: {np.std(mape_scores):.2f}%)")
print(f"  Average R2 Score: {np.mean(r2_scores):.2f} (Std: {np.std(r2_scores):.2f})")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.inspection import permutation_importance

# Ensure X_full and y_full are from the latest state (already defined in previous cells)

# Create a pipeline with StandardScaler and RandomForestRegressor
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('rf_regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

# Fit the pipeline on the full data
pipeline.fit(X_full, y_full)

# Calculate Permutation Feature Importance
# n_jobs=-1 uses all available CPU cores for faster computation
result = permutation_importance(
    pipeline, X_full, y_full, n_repeats=10, random_state=42, n_jobs=-1
)

# Organize the results into a pandas Series
sorted_features = pd.Series(
    result.importances_mean, index=X_full.columns
).sort_values(ascending=False)

print("Permutation Feature Importances for Random Forest Regressor (WQI Prediction):")
print(sorted_features)

# Plot the feature importances
plt.figure(figsize=(10, 6))
# Fix: Assign y to hue and set legend=False to resolve FutureWarning
sns.barplot(x=sorted_features.values, y=sorted_features.index, hue=sorted_features.index, palette='viridis', legend=False)
plt.title('Permutation Feature Importance for WQI Prediction (Random Forest Regressor)')
plt.xlabel('Mean Decrease in R2 Score (Importance)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

from sklearn.svm import SVR
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error
from sklearn.model_selection import KFold
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# Calculate MAPE
def mean_absolute_percentage_error(y_true, y_pred):
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero: add a small epsilon to avoid NaN or Inf where y_true is 0
    return np.mean(np.abs((y_true - y_pred) / (y_true + np.finfo(float).eps))) * 100

# Prepare the full dataset for cross-validation
# Define features (X_full) directly using merged_df, assuming merged_df is already defined.
# Drop 'WQI' as 'WQI_Cat' is derived from it, and 'stn_no' and 'Month' are identifiers
X_full = merged_df.drop(['WQI', 'WQI_Cat', 'stn_no', 'Month'], axis=1)

# Target for regression (continuous WQI)
y_full = merged_df['WQI']

n_splits = 5 # Number of folds
kf = KFold(n_splits=n_splits, shuffle=True, random_state=42)

print(f"\n--- Performing {n_splits}-Fold Cross-Validation for Support Vector Regressor (Optimized) ---")

# Best parameters found previously by Bayesian Optimization:
# svr__C: 48.166643974424204
# svr__epsilon: 0.02043356826316902
# svr__gamma: 0.047720535788681455

# Define the pipeline for scaling and SVR with the best parameters
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('svr', SVR(kernel='rbf', C=48.166643974424204, gamma=0.047720535788681455, epsilon=0.02043356826316902))
])

# Assign the configured pipeline as the best_svr_model for evaluation
best_svr_model = pipe

mse_scores = []
rmse_scores = []
mae_scores = []
mape_scores = []
r2_scores = []

# Evaluate the best model using the KFold strategy to get all metrics and their standard deviations
for fold, (train_index, test_index) in enumerate(kf.split(X_full)):
    X_train_fold, X_test_fold = X_full.iloc[train_index], X_full.iloc[test_index]
    y_train_fold, y_test_fold = y_full.iloc[train_index], y_full.iloc[test_index]

    # The best_svr_model is a Pipeline, so it handles scaling internally
    # We fit it on each fold's training data and predict on test data
    best_svr_model.fit(X_train_fold, y_train_fold)
    y_pred_fold = best_svr_model.predict(X_test_fold)

    # Evaluate the model for this fold
    mse_scores.append(mean_squared_error(y_test_fold, y_pred_fold))
    rmse_scores.append(np.sqrt(mean_squared_error(y_test_fold, y_pred_fold)))
    mae_scores.append(mean_absolute_error(y_test_fold, y_pred_fold))
    mape_scores.append(mean_absolute_percentage_error(y_test_fold, y_pred_fold))
    r2_scores.append(r2_score(y_test_fold, y_pred_fold))

# Calculate and print average metrics across all folds
print(f"\n--- Performance of Best SVR Model with {n_splits}-Fold Cross-Validation ---")
print(f"  Average MSE: {np.mean(mse_scores):.2f} (Std: {np.std(mse_scores):.2f})")
print(f"  Average RMSE: {np.mean(rmse_scores):.2f} (Std: {np.std(rmse_scores):.2f})")
print(f"  Average MAE: {np.mean(mae_scores):.2f} (Std: {np.std(mae_scores):.2f})")
print(f"  Average MAPE: {np.mean(mape_scores):.2f}% (Std: {np.std(mape_scores):.2f}%)")
print(f"  Average R2 Score: {np.mean(r2_scores):.2f} (Std: {np.std(r2_scores):.2f})")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.inspection import permutation_importance

# Ensure X_full and y_full are defined (as they should be from previous cells)
# X_full = merged_df.drop(['WQI', 'WQI_Cat', 'stn_no', 'Month'], axis=1)
# y_full = merged_df['WQI']

# Retrain the best SVR model (pipeline) on the entire dataset
# The 'best_svr_model' pipeline was defined in cell OpDrD01kvXlV
# It includes a StandardScaler and the optimized SVR.

best_svr_model.fit(X_full, y_full)

# Calculate Permutation Feature Importance
# n_repeats: number of times to permute a feature
# random_state for reproducibility
# n_jobs=-1 uses all available CPU cores
result = permutation_importance(
    best_svr_model, X_full, y_full, n_repeats=10, random_state=42, n_jobs=-1
)

# Organize the results into a pandas Series
sorted_importances = pd.Series(
    result.importances_mean, index=X_full.columns
).sort_values(ascending=False)

print("Permutation Feature Importances for SVR (WQI Prediction):")
print(sorted_importances)

# Plot the feature importances
plt.figure(figsize=(10, 6))
sns.barplot(x=sorted_importances.values, y=sorted_importances.index, palette='magma')
plt.title('Permutation Feature Importance for WQI Prediction (SVR)')
plt.xlabel('Mean Decrease in R2 Score (Importance)')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()


### Manual Parameter Importance: Correlation Analysis

To understand the importance of each parameter from a direct, univariate perspective, we can calculate the Pearson correlation coefficient between each water quality parameter and the Water Quality Index (WQI). A higher absolute correlation value indicates a stronger linear relationship. While model-based importances consider multivariate effects, direct correlation can provide a straightforward initial insight.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Ensure X_full and y_full are defined from previous cells
# X_full contains the water quality parameters
# y_full contains the continuous WQI values

# Combine features (X_full) and target (y_full) into a single DataFrame for correlation calculation
correlation_data = pd.concat([X_full, y_full], axis=1)

# Calculate the Pearson correlation of each feature with the WQI
wqi_correlations = correlation_data.corr()['WQI'].drop('WQI')

# Sort the correlations by their absolute value to see the most impactful features
sorted_wqi_correlations = wqi_correlations.abs().sort_values(ascending=False)

print("Absolute Pearson Correlation of each parameter with WQI:")
display(sorted_wqi_correlations)

# Visualize the correlations
plt.figure(figsize=(10, 6))
sns.barplot(x=sorted_wqi_correlations.values, y=sorted_wqi_correlations.index, palette='coolwarm')
plt.title('Absolute Pearson Correlation with WQI')
plt.xlabel('Absolute Correlation Coefficient')
plt.ylabel('Water Quality Parameter')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Combine features (X_full) and target (y_full) into a single DataFrame for correlation calculation
# X_full and y_full have been updated in the previous cell with the new WQI.
correlation_data_new_wqi = pd.concat([X_full, y_full], axis=1)

# Calculate the Pearson correlation of each feature with the NEW WQI
wqi_correlations_new = correlation_data_new_wqi.corr()['WQI'].drop('WQI')

# Sort the correlations by their absolute value to see the most impactful features
sorted_wqi_correlations_new = wqi_correlations_new.abs().sort_values(ascending=False)

print("Absolute Pearson Correlation of each parameter with NEW WQI (Custom Weights):")
display(sorted_wqi_correlations_new)

# Visualize the correlations
plt.figure(figsize=(10, 6))
sns.barplot(x=sorted_wqi_correlations_new.values, y=sorted_wqi_correlations_new.index, palette='coolwarm')
plt.title('Absolute Pearson Correlation with NEW WQI (Custom Weights)')
plt.xlabel('Absolute Correlation Coefficient')
plt.ylabel('Water Quality Parameter')
plt.tight_layout()
plt.show()

### Exploring the Distribution and WQI Impact of SAR and RSC

Given that SAR and RSC are highlighted as key parameters by IS standards, and that non-linear models (GPR, SVR) found them more important than linear models, let's examine their distribution within the dataset and how they relate to the calculated WQI. This can help us understand if their values are concentrated in ranges that do not strongly differentiate the WQI in a linear fashion, yet might have a critical non-linear influence.

We will visualize:
1.  The distribution of SAR and RSC values.
2.  Scatter plots showing SAR vs. WQI and RSC vs. WQI, possibly colored by WQI categories to highlight thresholds.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Set up the figure and axes for subplots
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
fig.suptitle('Distribution and WQI Relationship for SAR and RSC', fontsize=16)

# Plot 1: Distribution of SAR
sns.histplot(merged_df['SAR'], kde=True, ax=axes[0, 0], color='skyblue')
axes[0, 0].set_title('Distribution of SAR')
axes[0, 0].set_xlabel('SAR Value')
axes[0, 0].set_ylabel('Frequency')
axes[0, 0].axvline(x=10, color='r', linestyle='--', label='IS Good Threshold (SAR < 10)')
axes[0, 0].legend()

# Plot 2: SAR vs. WQI
sns.scatterplot(x=merged_df['SAR'], y=merged_df['WQI'], hue=merged_df['WQI_Cat'],
                palette='viridis', ax=axes[0, 1], alpha=0.7)
axes[0, 1].set_title('SAR vs. WQI')
axes[0, 1].set_xlabel('SAR Value')
axes[0, 1].set_ylabel('WQI Score')
axes[0, 1].axvline(x=10, color='r', linestyle='--', label='IS Good Threshold (SAR < 10)')
axes[0, 1].legend(title='WQI Category')

# Plot 3: Distribution of RSC
sns.histplot(merged_df['RSC'], kde=True, ax=axes[1, 0], color='lightgreen')
axes[1, 0].set_title('Distribution of RSC')
axes[1, 0].set_xlabel('RSC Value')
axes[1, 0].set_ylabel('Frequency')
axes[1, 0].axvline(x=2.5, color='r', linestyle='--', label='IS Good Threshold (RSC < 2.5)')
axes[1, 0].legend()

# Plot 4: RSC vs. WQI
sns.scatterplot(x=merged_df['RSC'], y=merged_df['WQI'], hue=merged_df['WQI_Cat'],
                palette='magma', ax=axes[1, 1], alpha=0.7)
axes[1, 1].set_title('RSC vs. WQI')
axes[1, 1].set_xlabel('RSC Value')
axes[1, 1].set_ylabel('WQI Score')
axes[1, 1].axvline(x=2.5, color='r', linestyle='--', label='IS Good Threshold (RSC < 2.5)')
axes[1, 1].legend(title='WQI Category')

plt.tight_layout(rect=[0, 0.03, 1, 0.96]) # Adjust layout to prevent title overlap
plt.show()

### Multi-Method Feature Importance Aggregation

To get a holistic view of parameter importance, we'll aggregate the feature importances derived from different regression models (Linear Regression, Bayesian Ridge, Gaussian Process Regressor, Random Forest Regressor, SVR) and Pearson correlation. By combining these, we can identify parameters that consistently show high importance across various analytical approaches.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Aggregate all feature importances into a dictionary
all_importances = {
    'Linear Regression Coeffs (Abs)': sorted_features_coef_lr_new_wqi,
    'Bayesian Ridge Coeffs (Abs)': sorted_features_coef_br_new_wqi,
    'GPR Permutation Importance': sorted_importances_gpr,
    'Random Forest Permutation Importance': sorted_features, # From cell IyAI-SjorCi-
    'SVR Permutation Importance': sorted_importances, # From cell QRMAO0GVrw8x
    'Pearson Correlation (Abs)': sorted_wqi_correlations_new
}

# Create a DataFrame from the dictionary
# Use .fillna(0) for any model that might not have a feature present in another
importance_df = pd.DataFrame(all_importances).fillna(0)

# Normalize each column so that the highest importance in each method is 1
# This allows for comparison across different scales of importance metrics
normalized_importance_df = importance_df.apply(lambda x: x / x.max(), axis=0)

print("Normalized Feature Importances Across Multiple Methods:")
display(normalized_importance_df.round(3))

# Plotting the aggregated feature importances
plt.figure(figsize=(12, 8))
sns.heatmap(normalized_importance_df.T, annot=True, cmap='viridis', fmt=".2f", linewidths=.5)
plt.title('Normalized Feature Importances Across Multiple Methods')
plt.xlabel('Feature')
plt.ylabel('Importance Method')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

# Calculate the average normalized importance for each feature
average_importance = normalized_importance_df.mean(axis=1).sort_values(ascending=False)

print("\nAverage Normalized Feature Importance:")
display(average_importance.round(3))

# Plot the average normalized importance
plt.figure(figsize=(10, 6))
sns.barplot(x=average_importance.values, y=average_importance.index, palette='magma')
plt.title('Average Normalized Feature Importance Across All Methods')
plt.xlabel('Average Normalized Importance')
plt.ylabel('Feature')
plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

# Data extracted from the standard outputs of previous cells
results = {
    'Linear Regression': {
        'R2': 0.77,
        'RMSE': 5.02,
        'MAE': 3.90,
        'MAPE': 5.65
    },
    'Bayesian Ridge Regression': {
        'R2': 0.77,
        'RMSE': 5.02,
        'MAE': 3.89,
        'MAPE': 5.64
    },
    'Gaussian Process Regressor': {
        'R2': 0.76,
        'RMSE': 4.57,
        'MAE': 2.16,
        'MAPE': 3.10
    },
    'Random Forest Regressor': {
        'R2': 0.91,
        'RMSE': 3.14,
        'MAE': 2.09,
        'MAPE': 3.25
    },
    'Support Vector Regressor (Optimized)': {
        'R2': 0.94,
        'RMSE': 2.53,
        'MAE': 1.71,
        'MAPE': 2.55
    }
}

# Create a DataFrame from the results
metrics_df = pd.DataFrame.from_dict(results, orient='index')
metrics_df.index.name = 'Model'

print("Regression Model Performance Comparison (Average Metrics):")
display(metrics_df.round(2))


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from imblearn.over_sampling import SMOTE

# Assuming X_train_clf, X_test_clf, y_train_clf, y_test_clf, and label_encoder are already defined.
# If not, please ensure the cell 'dVGs5lK-F2KM' has been run.

print("--- Training Random Forest Classifier for WQI Categories ---")

# Before applying SMOTE, let's check the class distribution in y_train_clf
print(f"Original y_train_clf class distribution: {np.unique(y_train_clf, return_counts=True)}")

# Apply SMOTE to balance the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_clf, y_train_clf)

print(f"Resampled y_train_clf class distribution: {np.unique(y_train_resampled, return_counts=True)}")

# Initialize and train the Random Forest Classifier with resampled data
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)
rf_classifier.fit(X_train_resampled, y_train_resampled)

# Make predictions on the test set
y_pred_clf = rf_classifier.predict(X_test_clf)

# Evaluate the classifier
accuracy = accuracy_score(y_test_clf, y_pred_clf)
clf_report = classification_report(y_test_clf, y_pred_clf, target_names=label_encoder.classes_)
conf_matrix = confusion_matrix(y_test_clf, y_pred_clf)

print(f"\nAccuracy: {accuracy:.2f}")
print("\nClassification Report:")
print(clf_report)

print("\nConfusion Matrix:")
display(pd.DataFrame(conf_matrix, index=label_encoder.classes_, columns=label_encoder.classes_))

# --- Draw Comparison for Agricultural Suitability ---

# Get original category labels for predictions and true values
y_test_labels = label_encoder.inverse_transform(y_test_clf)
y_pred_labels = label_encoder.inverse_transform(y_pred_clf)

# Create a DataFrame for easy comparison
comparison_df = pd.DataFrame({'True_Category': y_test_labels, 'Predicted_Category': y_pred_labels})

# Map categories to suitability
def map_suitability(category):
    if category == 'G':
        return 'Suitable'
    else:
        return 'Not Suitable'

comparison_df['True_Suitability'] = comparison_df['True_Category'].apply(map_suitability)
comparison_df['Predicted_Suitability'] = comparison_df['Predicted_Category'].apply(map_suitability)

# Count occurrences of true vs. predicted suitability
true_suitability_counts = comparison_df['True_Suitability'].value_counts()
predicted_suitability_counts = comparison_df['Predicted_Suitability'].value_counts()

# Plotting the comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
fig.suptitle('Agricultural Suitability Comparison (True vs. Predicted)', fontsize=16)

# True Suitability Plot
sns.barplot(x=true_suitability_counts.index, y=true_suitability_counts.values, ax=axes[0], palette='viridis')
axes[0].set_title('True Suitability')
axes[0].set_xlabel('Suitability')
axes[0].set_ylabel('Number of Samples')

# Predicted Suitability Plot
sns.barplot(x=predicted_suitability_counts.index, y=predicted_suitability_counts.values, ax=axes[1], palette='viridis')
axes[1].set_title('Predicted Suitability')
axes[1].set_xlabel('Suitability')
axes[1].set_ylabel('Number of Samples') # Shared y-axis

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("\nSummary of Agricultural Suitability:")
print("True Suitability Distribution:")
print(true_suitability_counts)
print("\nPredicted Suitability Distribution:")
print(predicted_suitability_counts)


### Gradient Boosting Classifier for WQI Categories

Let's evaluate the performance of a Gradient Boosting Classifier for predicting WQI categories and compare its suitability predictions.

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from imblearn.over_sampling import SMOTE

print("--- Training Gradient Boosting Classifier for WQI Categories ---")

# Before applying SMOTE, let's check the class distribution in y_train_clf
print(f"Original y_train_clf class distribution: {np.unique(y_train_clf, return_counts=True)}")

# Apply SMOTE to balance the training data
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_clf, y_train_clf)

print(f"Resampled y_train_clf class distribution: {np.unique(y_train_resampled, return_counts=True)}")

# Initialize and train the Gradient Boosting Classifier with resampled data
gb_classifier = GradientBoostingClassifier(n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42)
gb_classifier.fit(X_train_resampled, y_train_resampled)

# Make predictions on the original (unresampled) test set
y_pred_gb = gb_classifier.predict(X_test_clf)

# Evaluate the classifier
accuracy_gb = accuracy_score(y_test_clf, y_pred_gb)
clf_report_gb = classification_report(y_test_clf, y_pred_gb, target_names=label_encoder.classes_)
conf_matrix_gb = confusion_matrix(y_test_clf, y_pred_gb)

print(f"\nAccuracy: {accuracy_gb:.2f}")
print("\nClassification Report:")
print(clf_report_gb)

print("\nConfusion Matrix:")
display(pd.DataFrame(conf_matrix_gb, index=label_encoder.classes_, columns=label_encoder.classes_))

# --- Draw Comparison for Agricultural Suitability ---

# Get original category labels for predictions
y_test_labels_gb = label_encoder.inverse_transform(y_test_clf) # Use y_test_clf as the true labels for suitability
y_pred_labels_gb = label_encoder.inverse_transform(y_pred_gb)

# Create a DataFrame for easy comparison
comparison_df_gb = pd.DataFrame({'True_Category': y_test_labels_gb, 'Predicted_Category': y_pred_labels_gb})

# Map categories to suitability
def map_suitability(category):
    if category == 'G':
        return 'Suitable'
    else:
        return 'Not Suitable'

comparison_df_gb['True_Suitability'] = comparison_df_gb['True_Category'].apply(map_suitability)
comparison_df_gb['Predicted_Suitability'] = comparison_df_gb['Predicted_Category'].apply(map_suitability)

# Count occurrences of true vs. predicted suitability
true_suitability_counts_gb = comparison_df_gb['True_Suitability'].value_counts()
predicted_suitability_counts_gb = comparison_df_gb['Predicted_Suitability'].value_counts()

# Plotting the comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
fig.suptitle('Agricultural Suitability Comparison (True vs. Predicted) - Gradient Boosting', fontsize=16)

# True Suitability Plot
sns.barplot(x=true_suitability_counts_gb.index, y=true_suitability_counts_gb.values, ax=axes[0], palette='viridis')
axes[0].set_title('True Suitability')
axes[0].set_xlabel('Suitability')
axes[0].set_ylabel('Number of Samples')

# Predicted Suitability Plot
sns.barplot(x=predicted_suitability_counts_gb.index, y=predicted_suitability_counts_gb.values, ax=axes[1], palette='viridis')
axes[1].set_title('Predicted Suitability')
axes[1].set_xlabel('Suitability')
axes[1].set_ylabel('Number of Samples') # Shared y-axis

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("\nSummary of Agricultural Suitability (Gradient Boosting):")
print("True Suitability Distribution:")
print(true_suitability_counts_gb)
print("\nPredicted Suitability Distribution:")
print(predicted_suitability_counts_gb)

### Logistic Regression Classifier for WQI Categories

Let's evaluate the performance of a Logistic Regression Classifier for predicting WQI categories and compare its suitability predictions.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

print("--- Training Logistic Regression Classifier for WQI Categories ---")

# Initialize and train the Logistic Regression Classifier
# Setting max_iter to a higher value for convergence
logistic_classifier = LogisticRegression(max_iter=1000, random_state=42)
logistic_classifier.fit(X_train_clf, y_train_clf)

# Make predictions on the test set
y_pred_lr = logistic_classifier.predict(X_test_clf)

# Evaluate the classifier
accuracy_lr = accuracy_score(y_test_clf, y_pred_lr)
clf_report_lr = classification_report(y_test_clf, y_pred_lr, target_names=label_encoder.classes_)
conf_matrix_lr = confusion_matrix(y_test_clf, y_pred_lr)

print(f"\nAccuracy: {accuracy_lr:.2f}")
print("\nClassification Report:")
print(clf_report_lr)

print("\nConfusion Matrix:")
display(pd.DataFrame(conf_matrix_lr, index=label_encoder.classes_, columns=label_encoder.classes_))

# --- Draw Comparison for Agricultural Suitability ---

# Get original category labels for predictions
y_pred_labels_lr = label_encoder.inverse_transform(y_pred_lr)

# Create a DataFrame for easy comparison
comparison_df_lr = pd.DataFrame({'True_Category': y_test_labels, 'Predicted_Category': y_pred_labels_lr})

# Map categories to suitability
def map_suitability(category):
    if category == 'G':
        return 'Suitable'
    else:
        return 'Not Suitable'

comparison_df_lr['True_Suitability'] = comparison_df_lr['True_Category'].apply(map_suitability)
comparison_df_lr['Predicted_Suitability'] = comparison_df_lr['Predicted_Category'].apply(map_suitability)

# Count occurrences of true vs. predicted suitability
true_suitability_counts_lr = comparison_df_lr['True_Suitability'].value_counts()
predicted_suitability_counts_lr = comparison_df_lr['Predicted_Suitability'].value_counts()

# Plotting the comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
fig.suptitle('Agricultural Suitability Comparison (True vs. Predicted) - Logistic Regression', fontsize=16)

# True Suitability Plot
sns.barplot(x=true_suitability_counts_lr.index, y=true_suitability_counts_lr.values, ax=axes[0], palette='viridis')
axes[0].set_title('True Suitability')
axes[0].set_xlabel('Suitability')
axes[0].set_ylabel('Number of Samples')

# Predicted Suitability Plot
sns.barplot(x=predicted_suitability_counts_lr.index, y=predicted_suitability_counts_lr.values, ax=axes[1], palette='viridis')
axes[1].set_title('Predicted Suitability')
axes[1].set_xlabel('Suitability')
axes[1].set_ylabel('Number of Samples') # Shared y-axis

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()

print("\nSummary of Agricultural Suitability (Logistic Regression):")
print("True Suitability Distribution:")
print(true_suitability_counts_lr)
print("\nPredicted Suitability Distribution:")
print(predicted_suitability_counts_lr)


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Calculate the correlation matrix for the 6 parameters (X_full)
correlation_matrix = X_full.corr()

print("Correlation Matrix for the 6 Parameters:")
display(correlation_matrix)

# Visualize the correlation matrix using a heatmap
plt.figure(figsize=(8, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Correlation Matrix of Water Quality Parameters')
plt.show()